# 02 — Evaluation

Loads generated stories and computes scores.
- `EVAL_MODE = "metrics"` → log-likelihood coherence + self-BLEU diversity
- `EVAL_MODE = "gemini"` → Gemini LLM-as-judge (requires `GEMINI_API_KEY`)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import config
from src.generation import load_stories
from src.evaluation.metrics import score_coherence_batch, score_diversity

In [ ]:
# ── Load all generated stories ────────────────────────────────────────────────
stories_by_schedule = {}
for sched_name in config.SCHEDULES:
    out_dir = config.results_dir(config.MODEL_NAME, sched_name)
    path = os.path.join(out_dir, "stories.jsonl")
    if os.path.exists(path):
        stories_by_schedule[sched_name] = load_stories(path)
        print(f"{sched_name}: {len(stories_by_schedule[sched_name])} stories")
    else:
        print(f"WARNING: {path} not found — run 01_generate.ipynb first")

In [ ]:
if config.EVAL_MODE == "metrics":
    # Load model for log-likelihood coherence
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        config.MODEL_NAME,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        model = model.to(device)
    model.eval()

    # Score coherence per story
    for sched_name, stories in stories_by_schedule.items():
        print(f"Scoring coherence: {sched_name}")
        score_coherence_batch(model, tokenizer, stories)

    # Score diversity per schedule
    diversity_scores = score_diversity(stories_by_schedule)
    print("\nDiversity (self-BLEU, lower=more diverse):")
    for sched, score in diversity_scores.items():
        print(f"  {sched}: {score:.4f}")

elif config.EVAL_MODE == "gemini":
    from src.evaluation.llm_judge import score_stories_batch
    for sched_name, stories in stories_by_schedule.items():
        print(f"Gemini judging: {sched_name}")
        score_stories_batch(stories)

In [ ]:
# ── Save scores ───────────────────────────────────────────────────────────────
for sched_name, stories in stories_by_schedule.items():
    out_dir = config.results_dir(config.MODEL_NAME, sched_name)
    scores_path = os.path.join(out_dir, "scores.jsonl")
    with open(scores_path, "w") as f:
        for s in stories:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"Saved → {scores_path}")

# Save diversity separately (schedule-level)
if config.EVAL_MODE == "metrics":
    div_path = os.path.join(config.RESULTS_DIR, "diversity.json")
    with open(div_path, "w") as f:
        json.dump(diversity_scores, f, indent=2)
    print(f"Saved diversity → {div_path}")